<a href="https://www.kaggle.com/code/avikdas567/stanford-rna-3d-folding-part-2?scriptVersionId=294025866" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ============================================================
# Stanford RNA 3D Folding Part 2 — Improved Heuristic Baseline
# ============================================================

import os
import math
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

# -------------------------
# Config
# -------------------------
DATA_ROOT = "/kaggle/input/stanford-rna-3d-folding-2"
MSA_DIR = f"{DATA_ROOT}/MSA"
TEST_SEQ_PATH = f"{DATA_ROOT}/test_sequences.csv"
SAMPLE_SUB_PATH = f"{DATA_ROOT}/sample_submission.csv"
OUT_PATH = "submission.csv"

np.random.seed(42)
random.seed(42)

# -------------------------
# Load data
# -------------------------
test_df = pd.read_csv(TEST_SEQ_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

# -------------------------
# MSA utilities
# -------------------------
def load_msa(target_id):
    path = os.path.join(MSA_DIR, f"{target_id}.MSA.fasta")
    if not os.path.exists(path):
        return None
    seqs = []
    with open(path) as f:
        for line in f:
            if not line.startswith(">"):
                seqs.append(line.strip())
    return seqs

def conservation(msa):
    """Simple per-position conservation score"""
    L = len(msa[0])
    cons = np.zeros(L)
    for i in range(L):
        col = [s[i] for s in msa if s[i] != "-"]
        if len(col) == 0:
            continue
        freqs = {}
        for c in col:
            freqs[c] = freqs.get(c, 0) + 1
        cons[i] = max(freqs.values()) / len(col)
    return cons

# -------------------------
# Geometry utilities
# -------------------------
def helix(n, radius=7.5, rise=3.2, twist=32.7):
    coords = np.zeros((n, 3))
    for i in range(n):
        a = np.deg2rad(i * twist)
        coords[i] = [
            radius * np.cos(a),
            radius * np.sin(a),
            i * rise,
        ]
    return coords

def rotate(coords, axis, angle):
    axis = axis / np.linalg.norm(axis)
    a = np.cos(angle / 2)
    b, c, d = -axis * np.sin(angle / 2)
    rot = np.array([
        [a*a+b*b-c*c-d*d, 2*(b*c-a*d), 2*(b*d+a*c)],
        [2*(b*c+a*d), a*a+c*c-b*b-d*d, 2*(c*d-a*b)],
        [2*(b*d-a*c), 2*(c*d+a*b), a*a+d*d-b*b-c*c]
    ])
    return coords @ rot.T

def center(x):
    return x - x.mean(0, keepdims=True)

# -------------------------
# Build predictions
# -------------------------
rows = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    tid = row["target_id"]
    seq = row["sequence"]
    L = len(seq)

    msa = load_msa(tid)
    if msa is not None:
        cons = conservation(msa)
    else:
        cons = np.ones(L) * 0.5

    # classify positions
    is_stem = cons > 0.7

    # build backbone
    base = np.zeros((L, 3))
    i = 0
    cursor = np.zeros(3)
    direction = np.array([0, 0, 1])

    while i < L:
        if is_stem[i]:
            j = i
            while j < L and is_stem[j]:
                j += 1
            h = helix(j - i)
            h = rotate(h, np.array([1,0,0]), random.uniform(0, math.pi))
            h += cursor
            base[i:j] = h
            cursor = h[-1]
            i = j
        else:
            # loop: random walk
            base[i] = cursor + np.random.normal(scale=4.0, size=3)
            cursor = base[i]
            i += 1

    base = center(base)

    # generate 5 diverse global folds
    preds = []
    for k in range(5):
        conf = base.copy()
        axis = np.random.normal(size=3)
        angle = np.random.uniform(0, math.pi)
        conf = rotate(conf, axis, angle)
        conf += np.random.normal(scale=1.0 + 0.3*k, size=conf.shape)
        conf = np.clip(conf, -999.999, 9999.999)
        preds.append(conf)

    # write rows
    for i, nt in enumerate(seq):
        out = {
            "ID": f"{tid}_{i+1}",
            "resname": nt,
            "resid": i + 1,
        }
        for k in range(5):
            out[f"x_{k+1}"] = preds[k][i,0]
            out[f"y_{k+1}"] = preds[k][i,1]
            out[f"z_{k+1}"] = preds[k][i,2]
        rows.append(out)

# -------------------------
# Save submission
# -------------------------
sub = pd.DataFrame(rows)
sub = sub[list(sample_sub.columns)]
sub.to_csv(OUT_PATH, index=False)

print("submission.csv written")
display(sub.head())

100%|██████████| 28/28 [00:01<00:00, 18.21it/s]


submission.csv written


,ID,resname,resid,x_1,y_1,z_1,x_2,y_2,z_2,x_3,y_3,z_3,x_4,y_4,z_4,x_5,y_5,z_5
0,8ZNQ_1,A,1,-11.228970,3.521928,-5.317189,-10.258847,-0.320300,-7.796433,-13.328355,3.699488,-0.345322,-2.247116,1.492786,14.136551,7.608600,3.921750,-13.484806
1,8ZNQ_2,C,2,-4.709446,2.443526,-6.227378,-5.011127,2.354807,-7.776715,-5.535343,4.609528,-5.704874,2.508391,-0.703393,7.420118,4.590517,6.101671,-3.119528
2,8ZNQ_3,C,3,1.481983,6.515047,-5.113494,2.632533,3.386433,-9.677511,4.241204,10.243706,-2.975666,7.091952,1.668672,2.842056,5.619596,5.149464,-2.420441
3,8ZNQ_4,G,4,5.942392,5.175357,-6.056110,5.771980,1.653870,-9.853494,4.067528,5.732898,-6.375270,7.636453,1.590741,1.616190,10.924554,5.846563,3.579576
4,8ZNQ_5,U,5,12.077701,3.275442,-6.362769,12.838415,1.007222,-4.508058,11.902220,5.148097,-7.585708,11.272397,-0.122421,-5.474373,7.052497,-0.455475,10.211936
